# Den 4: Zpracování textových dat – Cvičení: Trénování Word2Vec & Klasifikace

## Zadání úkolu:
1. Načtěte dataset `imdb_reviews_preprocessed.csv` pomocí knihovny **Pandas**.
2. Na základě sloupce `review_tokens_lemmatized` natrénujte model **Word2Vec** pomocí knihovny **Gensim**. Zvolte vhodné hyperparametry (počet epoch, minimální počet výskytů slova, dimenzi vektoru, CBOW vs. Skip-gram).
3. Pro slova `cast`, `movie`, `comedy`, `watch` a `interesting` vraťte **10 nejpodobnějších slov** pro každé z nich.
4. Pomocí vhodné metody vypočítejte podobnost mezi slovy `movie` a `comedy`.
5. Zobrazte vektor pro slovo `show`.
6. Pro každou recenzi vygenerujte **průměrný Word2Vec vektor** (Sentence Embedding) pomocí metody `.apply()` a lambda výrazu.
7. Pomocí modelu **Logistické regrese** vytvořte klasifikátor pozitivního a negativního sentimentu.
8. Natrénujte algoritmus na trénovacích datech a predikujte na testovací sadě.
9. Vygenerujte `classification_report` a zhodnoťte výsledky modelu.

### Krok 1: Import knihoven a načtení dat

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from gensim.models import Word2Vec
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

data_path = Path("../data/imdb_reviews_preprocessed.csv")
if not data_path.exists():
    data_path = Path("data/imdb_reviews_preprocessed.csv")

imdb_reviews = pd.read_csv(data_path)
text_col = "review_tokens_lemmatized" if "review_tokens_lemmatized" in imdb_reviews.columns else "review_lemmatized"
print(f"Načteno recenzí: {len(imdb_reviews)}")
imdb_reviews.head()

### Krok 2: Příprava vět a trénování modelu Word2Vec (Gensim)
Vytvoříme seznam tokenizovaných vět (`list of lists`) a natrénujeme model CBOW (`sg=0`) s dimenzí `vector_size=100`, kontextovým oknem `window=5`, prahem `min_count=3` a `epochs=15`.

In [ ]:
sentences = [str(text).split() for text in imdb_reviews[text_col].fillna("")]

print("Trénuji Word2Vec model...")
w2v_model = Word2Vec(
    sentences=sentences,
    vector_size=100,
    window=5,
    min_count=3,
    workers=4,
    sg=0,  # CBOW
    epochs=15,
    seed=42
)

print(f"Model natrénován! Velikost slovníku: {len(w2v_model.wv):,} slov.")

### Krok 3: TOP 10 nejpodobnějších slov pro [cast, movie, comedy, watch, interesting]

In [ ]:
target_words = ["cast", "movie", "comedy", "watch", "interesting"]

for word in target_words:
    print(f"=== TOP 10 NEJPODOBNĚJŠÍCH SLOV K '{word}' ===")
    similar = w2v_model.wv.most_similar(word, topn=10)
    for rank, (sim_word, score) in enumerate(similar, 1):
        print(f"{rank:2d}. {sim_word:<15} (kosinová podobnost: {score:.4f})")
    print()

### Krok 4: Výpočet podobnosti mezi slovy `movie` a `comedy`

In [ ]:
similarity = w2v_model.wv.similarity("movie", "comedy")
print(f"Podobnost mezi 'movie' a 'comedy': {similarity:.4f}")

### Krok 5: Zobrazení vektoru pro slovo `show`

In [ ]:
show_vector = w2v_model.wv.get_vector("show")
print(f"Tvar vektoru 'show': {show_vector.shape}")
print("Prvních 10 komponent vektoru:")
print(show_vector[:10])

### Krok 6: Výpočet průměrného vektoru pro každou recenzi pomocí `.apply()` a lambdy

In [ ]:
def get_review_vector(text, model):
    words = [w for w in str(text).split() if w in model.wv]
    if not words:
        return np.zeros(model.vector_size, dtype=np.float32)
    return np.mean([model.wv[w] for w in words], axis=0).astype(np.float32)

print("Počítám průměrné Word2Vec vektory recenzí přes .apply()...")
imdb_reviews["review_vector"] = imdb_reviews[text_col].apply(
    lambda text: get_review_vector(text, w2v_model)
)
print("Vektory úspěšně spočteny!")

### Krok 7 & 8: Příprava dat a trénování Logistické regrese (max_iter=1000)

In [ ]:
X = np.vstack(imdb_reviews["review_vector"].values)
y = imdb_reviews["sentiment"].apply(lambda s: 1 if str(s).lower() == "positive" else 0)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Trénovací matice X: {X_train.shape}")
clf = LogisticRegression(max_iter=1000, random_state=42)
clf.fit(X_train, y_train)
print("Model úspěšně natrénován!")

### Krok 9: Predikce na testovací sadě a Classification Report

In [ ]:
y_pred = clf.predict(X_test)
print(f"Celková přesnost (Accuracy): {accuracy_score(y_test, y_pred)*100:.2f} %\n")
print(classification_report(y_test, y_pred, target_names=["Negative (0)", "Positive (1)"]))